<a href="https://colab.research.google.com/github/Song-yiJung/jggk-museum-catalog/blob/main/%EC%A1%B0%EC%84%A0%EC%B4%9D%EB%8F%85%EB%B6%80%EB%B0%95%EB%AC%BC%EA%B4%80%EB%AC%B8%EC%84%9C_%EC%9B%B9%EC%8A%A4%ED%81%AC%EB%9E%98%ED%95%91.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 조선총독부박물관 문서 서지 목록 수집

국립중앙박물관 「조선총독부박물관 문서」 상세페이지를 수집하여
문서철 · 문건 · 문서 세 계층의 서지 정보를 정리한다.

- 대상 `https://www.museum.go.kr/modern-history/`
- 산출 문서철 602 · 문건 6,568 · 문서 41,913 · 총 116,983면
- 최초 수집 2025년 11월

## 수집 구조

```
STEP 1   대분류별 권(문서철) 목록 수집
STEP 2   pseq ID 탐색 — 유효한 상세페이지 식별
STEP 3   유효 ID 대상 전체 페이지 수집
STEP 4   문서 상세 + 문건 메타 병합
STEP 5   pseq별 breadcrumb 수집 (캐시 방식)
STEP 6   breadcrumb을 권 · 문건으로 분리
```

## 주의

- 서버 부하를 고려해 요청 간격과 동시 실행 수를 조절한다.
- 웹 자료이므로 수집 시점에 따라 결과가 달라질 수 있다.
- STEP 2와 STEP 5는 시간이 오래 걸린다. 각각 로그와 캐시로 중단·재개를 지원한다.

---
## 0. 준비

In [ ]:
!pip install -q requests beautifulsoup4 lxml pandas tqdm html5lib

In [ ]:
import os, re, time, pickle, warnings
from concurrent.futures import ThreadPoolExecutor

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.mode.chained_assignment = None

BASE_URL = "https://www.museum.go.kr/modern-history/"
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"),
    "Accept-Language": "ko-KR,ko;q=0.9",
    "Referer": "https://www.museum.go.kr/",
}

WORKDIR = "/content"          # Colab 기본. 로컬 실행 시 경로를 바꾼다.
os.makedirs(WORKDIR, exist_ok=True)

F_VOLUMES  = f"{WORKDIR}/01_권목록.csv"
F_DOCS     = f"{WORKDIR}/02_문서_상세.csv"
F_MERGED   = f"{WORKDIR}/03_통합.csv"
F_CACHE    = f"{WORKDIR}/breadcrumb_cache.pkl"
F_LOG      = f"{WORKDIR}/pseq_success_log.txt"
F_FINAL    = f"{WORKDIR}/조선총독부박물관문서_수집_전체목록.csv"

# 대분류. 「고적조사」는 하위 구분이 있어 num 파라미터로 나뉜다.
COMMON_CATEGORIES = ["기부","진열","구입","발견","지정","보존","국유림","도면","지도","기타"]
GOJEOKJOSA_SUBS = [
    {"name":"고적조사 전체",   "num":0},
    {"name":"고적조사위원회", "num":1},
    {"name":"복명서",        "num":2},
    {"name":"조사보고",      "num":3},
    {"name":"고적 유물 목록", "num":4},
    {"name":"사진도면",      "num":5},
]

# pseq 탐색 범위. 상한은 여유를 두고 잡는다.
START_PSEQ, END_PSEQ = 1, 8050

In [ ]:
def make_session():
    """재시도 정책을 적용한 세션."""
    s = requests.Session()
    retry = Retry(total=3, backoff_factor=1,
                  status_forcelist=[429, 500, 502, 503, 504])
    s.mount("http://",  HTTPAdapter(max_retries=retry))
    s.mount("https://", HTTPAdapter(max_retries=retry))
    return s


def get(url, timeout=15, retries=3, delay=1):
    """단순 재시도 요청."""
    for _ in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout)
            r.raise_for_status()
            return r
        except requests.Timeout:
            time.sleep(delay)
        except requests.RequestException:
            break
    return None

---
## STEP 1. 권(문서철) 목록 수집

대분류별 목록 페이지를 1페이지부터 순회하며 표를 가져온다.
표에 「권 제목」과 「번호」 컬럼이 있으면 목록 표로 판정한다.

In [ ]:
def scrape_volume_list(label, major, out, num=None):
    base = f"{BASE_URL}group.do?major={major}"
    if num is not None:
        base += f"&num={num}"

    page = 1
    while True:
        r = get(f"{base}&page={page}")
        if r is None:
            break
        try:
            tables = pd.read_html(r.text, flavor="lxml")
        except ValueError:
            break

        df = next((t for t in tables
                   if {"권 제목", "번호"}.issubset(t.columns)), None)
        if df is None or df.empty:
            break

        soup = BeautifulSoup(r.text, "html.parser")
        links = [BASE_URL + a.get("href") for a in soup.select("td.ltxt a")]

        df["카테고리"] = label
        df["상세 문서 목록 링크"] = pd.Series(links)
        df.rename(columns={"원문": "원문 상태"}, inplace=True)

        cols = ["카테고리","번호","권 제목","생산 연도","생산 부서","쪽수","상세 문서 목록 링크"]
        out.append(df[[c for c in cols if c in df.columns]].copy())

        page += 1
        time.sleep(0.1)


volumes = []
for major in tqdm(COMMON_CATEGORIES, desc="일반 대분류"):
    scrape_volume_list(major, major, volumes)

for sub in tqdm(GOJEOKJOSA_SUBS, desc="고적조사 세부"):
    scrape_volume_list(f"고적조사_{sub['name']}", "고적조사", volumes, num=sub["num"])

df_volumes = pd.concat(volumes, ignore_index=True)
df_volumes.to_csv(F_VOLUMES, index=False, encoding="utf-8-sig")
print(f"권 목록 {len(df_volumes)}건 저장 → {F_VOLUMES}")

---
## STEP 2. pseq ID 탐색

상세페이지는 `doc.do?pseq=N` 형식이며 N이 연속되지 않는다.
1부터 상한까지 요청해 유효한 것만 골라낸다.

성공한 ID를 로그에 기록하므로 중단 후 재실행하면 이어서 진행한다.

In [ ]:
def load_log(path):
    if os.path.exists(path):
        with open(path) as f:
            return {int(x.strip()) for x in f if x.strip().isdigit()}
    return set()


def append_log(path, pseq):
    with open(path, "a") as f:
        f.write(f"{pseq}\n")


def fetch_doc_page(pseq):
    """상세페이지 1페이지를 가져와 문서 목록 표를 반환한다. 없으면 None."""
    r = get(f"{BASE_URL}doc.do?pseq={pseq}&page=1", timeout=10)
    if r is None:
        return None
    try:
        tables = pd.read_html(r.text, flavor="lxml")
        df = next((t for t in tables
                   if {"번호", "문서 제목"}.issubset(t.columns)), None)
        if df is None or df.empty:
            return None

        soup = BeautifulSoup(r.text, "html.parser")
        sel_h2   = "#container > div > div:nth-child(2) > div:nth-child(1) > h2"
        sel_path = "#container > div > div:nth-child(2) > div:nth-child(1) > div"
        h2   = soup.select_one(sel_h2)
        path = soup.select_one(sel_path)

        df["pseq_ID"]       = str(pseq)
        df["문서_대제목"]     = h2.get_text(strip=True) if h2 else None
        df["상위_경로_정보"]  = (path.get_text(strip=True).replace("\n", " ")
                             if path else None)

        cols = ["pseq_ID","문서_대제목","상위_경로_정보",
                "번호","문서 제목","문서종류","쪽수"]
        return df[[c for c in cols if c in df.columns]].copy()
    except Exception:
        return None


done = load_log(F_LOG)
targets = [p for p in range(START_PSEQ, END_PSEQ + 1) if p not in done]
print(f"탐색 대상 {len(targets)}건 (완료 {len(done)}건 제외)")

found = []
with ThreadPoolExecutor(max_workers=10) as ex:
    results = list(tqdm(ex.map(fetch_doc_page, targets),
                        total=len(targets), desc="pseq 탐색"))

for pseq, df in zip(targets, results):
    if df is not None:
        found.append(df)
        append_log(F_LOG, pseq)

df_docs = pd.concat(found, ignore_index=True)
df_docs.to_csv(F_DOCS, index=False, encoding="utf-8-sig")
print(f"문서 {len(df_docs)}건 / 유효 pseq {df_docs['pseq_ID'].nunique()}건 → {F_DOCS}")

---
## STEP 3. breadcrumb 수집

문서철·문건 계층은 상세페이지의 breadcrumb에 있다.

```
Home>권 목록>기부품 목록> 山口源固 소장품 목록
              └─ 권 ─┘  └────── 문건 ──────┘
```

pseq 단위로 한 번만 요청하면 되므로 중복을 제거한 뒤 수집한다.
결과는 pickle 캐시에 저장해 재실행 시 재사용한다.

In [ ]:
def fetch_breadcrumb(url):
    s = make_session()
    try:
        r = s.get(url, headers=HEADERS, timeout=20)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        h2 = soup.find("h2", class_="text_map_title")
        bc = soup.find("div", class_="breadcrumb")

        return [
            h2.get_text(strip=True) if h2 else None,
            " ".join(bc.get_text(strip=True).split()) if bc else None,
        ]
    except Exception:
        return [None, None]
    finally:
        s.close()


def load_cache(path):
    try:
        with open(path, "rb") as f:
            c = pickle.load(f)
        print(f"캐시 로드 {len(c)}건")
        return c
    except FileNotFoundError:
        print("새 캐시 생성")
        return {}


cache = load_cache(F_CACHE)
uniq  = df_docs["pseq_ID"].dropna().unique()
todo  = [p for p in uniq if p not in cache]
print(f"전체 pseq {len(uniq)}건 / 신규 {len(todo)}건")

if todo:
    with ThreadPoolExecutor(max_workers=5) as ex:
        urls = [f"{BASE_URL}doc.do?pseq={p}" for p in todo]
        for pid, res in zip(todo, tqdm(ex.map(fetch_breadcrumb, urls),
                                       total=len(todo), desc="breadcrumb")):
            cache[pid] = res

    with open(F_CACHE, "wb") as f:
        pickle.dump(cache, f)
    print(f"캐시 저장 {len(cache)}건")

---
## STEP 4. 병합

In [ ]:
df_cache = pd.DataFrame(
    [(k, v[0], v[1]) for k, v in cache.items()],
    columns=["pseq_ID", "대분류", "breadcrumb"],
)

df = df_docs.merge(df_cache, on="pseq_ID", how="left")
df["full_url"] = BASE_URL + "doc.do?pseq=" + df["pseq_ID"].astype(str)

print(f"병합 {len(df)}건 / breadcrumb 결측 {df['breadcrumb'].isna().sum()}건")
df.to_csv(F_MERGED, index=False, encoding="utf-8-sig")

---
## STEP 5. breadcrumb 분리

`Home>권 목록>` 접두를 제거하고 `>` 로 나눈다.

- 2단이면 앞이 권, 뒤가 문건
- 1단이면 권만 있고 문건은 비운다

분리 결과가 소장기관의 공식 계층 구분과 일치하는지는 별도 확인이 필요하다.

In [ ]:
PREFIX = re.compile(r"^\s*Home\s*>\s*권\s*목록\s*>\s*")

def split_breadcrumb(s):
    if not isinstance(s, str) or not s.strip():
        return pd.Series([None, None, None])

    cleaned = PREFIX.sub("", s).strip()
    parts   = [p.strip() for p in cleaned.split(">") if p.strip()]

    if len(parts) >= 2:
        return pd.Series([cleaned, parts[0], parts[-1]])
    if len(parts) == 1:
        return pd.Series([cleaned, parts[0], None])
    return pd.Series([cleaned, None, None])


df[["정제", "권", "문건"]] = df["breadcrumb"].apply(split_breadcrumb)

print(f"권 결측 {df['권'].isna().sum()}건 / 문건 결측 {df['문건'].isna().sum()}건")
print(f"고유 권 {df['권'].nunique()} · 고유 문건명 {df['문건'].nunique()} · 고유 pseq {df['pseq_ID'].nunique()}")

---
## STEP 6. 정리 및 저장

In [ ]:
OUT_COLS = ["순번","대분류","문서종류","쪽수","full_url","breadcrumb",
            "정제","권","문건","문서 제목"]

df = df.reset_index(drop=True)
df["순번"] = df.index + 1
df["쪽수"] = pd.to_numeric(df["쪽수"], errors="coerce")

final = df[[c for c in OUT_COLS if c in df.columns]].copy()
final.to_csv(F_FINAL, index=False, encoding="utf-8-sig")

print(f"저장 완료 → {F_FINAL}")
print()
print(f"문서철(권)  {final['권'].nunique():>7,}")
print(f"문건(pseq) {df['pseq_ID'].nunique():>7,}")
print(f"문서        {len(final):>7,}")
print(f"총 면수      {int(final['쪽수'].sum()):>7,}")
print()
print(final["대분류"].value_counts().to_string())

In [ ]:
# Colab에서 내려받기
try:
    from google.colab import files
    files.download(F_FINAL)
except ImportError:
    pass

---
## 한계

- 상세페이지에 게시된 범위만 수집한다. 디지털화되지 않은 문서는 포함되지 않는다.
- 「권」·「문건」은 breadcrumb 문자열을 분리한 결과이며, 소장기관의 공식 계층
  구분과 일치하는지는 별도 확인이 필요하다.
- 쪽수는 상세페이지 표기값이다. 펼침 촬영이나 동일 낱장의 반복 촬영으로 인해
  실제 이미지 수와 다를 수 있다.
- 웹 자료이므로 수집 시점에 따라 결과가 달라진다. 인용 시 수집 시점을 밝힌다.

## 출처

국립중앙박물관 「조선총독부박물관 문서」
https://www.museum.go.kr/modern-history/

## 인용

```
정송이, 조선총독부박물관 문서 서지 목록, 2025.
https://github.com/Song-yiJung/jggk-museum-catalog
```